**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/catalystneuro/encore", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/kim-2024"], check=True)
    os.chdir("/content/encore/entries/kim-2024")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Kim et al. (2024) from dandiset 001544

Paper. Kim R, Ananth MR, Desai NS, Role LW, Talmage DA (2024). Distinct subpopulations of ventral
pallidal cholinergic projection neurons encode valence of olfactory stimuli. *Cell Reports* 43(4),
114009. [doi:10.1016/j.celrep.2024.114009](https://doi.org/10.1016/j.celrep.2024.114009). The
claims file was written from the NIH author manuscript in PubMed Central (PMC11080946), main text,
STAR Methods and figure legends; this attempt also read the supplement (Figures S1 to S14 and their
legends), which is now in the paper's text. After a question from this attempt, a reader read the
Figure S12 legend and amended figs12-other-features-shared and added figs12-app-differs-from-all;
both are tested here as amended.

Dandiset. [DANDI:001544](https://dandiarchive.org/dandiset/001544/0.260827.1410), version
0.260827.1410. The paper cites no DANDI version (the dandiset was made after publication). This is
the only published version, and the most recent; the draft has the same 44 assets and size, so
nothing in the draft is missing from it.

What the dandiset holds. Whole-cell current-clamp recordings of 44 VP cholinergic neurons,
one per file, each a family of 500 ms current steps: 12 neurons labelled as activated by the
appetitive (APP) odor, 8 by the aversive (AV) odor, and 24 from home-cage ChAT-tau-eGFP mice (ALL).
These are the data of Figure 6F to 6H and Figure S12. The behaviour, histology, fiber photometry,
chirp currents and neuronal morphology of the paper are not in it.

Code. Features follow the STAR Methods, with the paper's code
([nsdesai/patch_clamp_analysis](https://github.com/nsdesai/patch_clamp_analysis) at tag v1.0.0,
commit 4ee1125) where the Methods are silent, as the claims file says; each parameter's source is
in the docstring of `vpchat.py`.

| Claim | Tested here |
|---|---|
| fig6f-max-firing-rate | tested |
| figs12a-sag-higher | tested |
| figs12-other-features-shared | tested |
| figs12-app-differs-from-all | tested |
| fig6g-av-more-hyperpolarized | tested |
| fig6h-av-smaller-ahp | tested |
| fig6-app-less-excitable | tested |
| fig1c, figs1, fig2c, fig2e, fig5c, fig5e (behaviour) | blocked: data missing |
| fig1f, fig1g, fig3, fig4d, figs5, figs7, figs9, figs11, fig7d (histology) | blocked: data missing |
| fig1k, fig1l, fig1mn (fiber photometry) | blocked: data missing |
| fig6j, fig6kl, fig6m, fig6n, figs13 (morphology) | blocked: data missing |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import vpchat as vc

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore")

started = time.time()
cells = vc.load_all()
print(f"loaded {len(cells)} files from dandiset {vc.DANDISET_ID} version {vc.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
meta = pd.DataFrame([c["meta"] for c in cells])
meta["file"] = meta.path.str.split("/").str[-1]
print("Cells (files) by group, as session_description gives it:")
print(meta.group.value_counts().to_string())
print("\nMice by group:", meta.groupby("group").subject.nunique().to_dict())
print("Paper (Figure 6 legend): APP 12 cells from 6 mice (3 male, 3 female), AV 8 cells from 4 mice "
      "(2 male, 2 female), ALL 21 cells from 8 mice.")
sex = meta.drop_duplicates("subject").groupby(["group", "sex"]).size().unstack(fill_value=0)
print("\nMice by group and sex:")
print(sex.to_string())
print("\nGenotype by group:")
print(meta.groupby(["group", "genotype"]).size().to_string())
print("\nAge by mouse:")
print(meta.drop_duplicates("subject")[["group", "subject", "sex", "age"]].to_string(index=False))
print("\nCells per mouse:")
print(meta.groupby(["group", "subject"]).size().to_string())

The 12 APP cells from 6 mice and 8 AV cells from 4 mice match the Figure 6 legend; the AV mice are
2 male and 2 female, as stated, but the APP mice are 4 male and 2 female where the legend says 3
and 3. The dandiset holds 24 ALL cells from 11 mice, where the paper analysed 21 from 8; it does
not say which three cells or three mice were left out of the paper. Five of the ALL mice are 5.7 to
6.6 weeks old, younger than the 8 to 12 weeks of the STAR Methods.

In [ ]:
# Can the paper's 21 ALL cells from 8 mice be had by leaving out whole mice?
import itertools
all_mice = meta[meta.group == "ALL"].groupby("subject").size()
totals = [int(all_mice[list(c)].sum()) for c in itertools.combinations(all_mice.index, 8)]
print("ALL cells per mouse:", all_mice.to_dict())
print("Cells in any 8 of the 11 ALL mice:", min(totals), "to", max(totals),
      "; subsets of 8 mice holding 21 cells:", sum(t == 21 for t in totals))

No choice of 8 of the 11 ALL mice holds 21 cells: only two mice contribute a single cell, so
leaving out three mice removes at least four cells (8 mice hold 14 to 20 cells). The paper's 21
cells from 8 mice therefore cannot be the dandiset's cells less some whole mice; either cells were
left out within mice as well, or the counts of the legend differ from the deposit. The `no_FL` set
used below (21 cells from 10 mice) matches the cell count only.

In [ ]:
rows = []
for c in cells:
    m, I, V = c["meta"], c["I"], c["V"]
    start, stop, steps, order = vc.step_layout(c, use_all_steps=True)
    dt = 1000 / c["rate"]
    rows.append({
        "id": m["identifier"], "group": m["group"], "sweeps": m["n_sweeps"], "stim": m["n_stim"],
        "samples": V.shape[0], "rate_Hz": c["rate"], "stim_rate_Hz": m["stim_rates"][0],
        "sweep_starts": m["starting_times"], "V_unit": m["voltage_unit"], "I_unit": m["stimulus_unit"],
        "I_min": I.min(), "I_max": I.max(), "V_min_mV": round(float(np.nanmin(V)), 1),
        "V_max_mV": round(float(np.nanmax(V)), 1), "nan_samples": int(np.isnan(V).sum()),
        "step_on_ms": start * dt, "step_ms": (stop - start + 1) * dt,
        "steps_pA": ",".join(str(int(s)) for s in np.sort(steps)),
        "bias_A": m["bias_current"], "bridge_Ohm": m["bridge_balance"]})
info = pd.DataFrame(rows)
print(info.drop(columns="steps_pA").to_string(index=False))
print("\nStep families (pA):")
print(info.groupby("steps_pA").id.apply(list).to_string())
print("\nDistinct values over all files: bias_current", info.bias_A.unique().tolist(),
      "; bridge_balance", info.bridge_Ohm.unique().tolist())
print("Stimulus range over all files:", info.I_min.min(), "to", info.I_max.max(), "(unit stated:", info.I_unit.unique().tolist(), ")")
print("Voltage range over all files (mV):", info.V_min_mV.min(), "to", info.V_max_mV.max())

- Units. The stimulus series are stated in amperes with conversion 1, but hold values from -60 to
  500: they are picoamperes, the step amplitudes of the Methods. The voltage, stated in volts,
  ranges over values that are millivolts after conversion (resting near -65 mV, spike peaks near
  +25 mV), so its unit is right.
- Sampling. Every series is at 10 kHz. In 40 files a sweep lasts 2 s and the step starts at 200
  ms; in the three files whose identifier ends in `_FL` a sweep lasts 5 s and the step starts at
  1000 ms. Every step lasts 500 ms, as the Methods say, and every sweep carries a -20 pA, 100 ms
  test pulse after the step.
- Every sweep's `starting_time` is 0, so the 10 s between steps of the Methods cannot be checked.
- `bias_current` is 1e-11 A (10 pA) and `bridge_balance` 2e7 Ohm in every file. A single DC
  current cannot hold 44 cells at -65 mV, so these look like placeholders; the holding current is
  not recorded, and no segment of any sweep is free of it. The Methods' membrane potential
  "measured in the absence of a current injection" (reading (b) of the claims file) cannot be
  measured; only the pre-step potential under the holding current (reading (a)) can.
- The step families differ between cells: most run from -60 to 200 pA, some to 150 or 70 pA, and
  some ALL cells go to 300, 400 or 500 pA, with repeated families in three files. Only steps from
  -60 to 200 pA are used (Methods).
- Three sweeps end early: their voltage is NaN for the last 0.1 to 0.6 s, after the step.
- The voltage floor. Five files reach exactly -105.6 mV and stay there for 10 ms or more during
  the -60 pA step (see the sag claim): the recording saturates there, which truncates the response
  of cells with high input resistance.

In [ ]:
# The injected current before the step, and the potential of the 0 pA sweep, per cell: is any
# segment free of the holding current?
rows = []
for c in cells:
    start, stop, steps, order = vc.step_layout(c, use_all_steps=True)
    j0 = [k for k in order if steps[k] == 0]
    rows.append({"id": c["meta"]["identifier"], "group": c["meta"]["group"],
                 "max |I| before step (pA)": float(np.abs(c["I"][:start, :]).max()),
                 "0 pA sweep, mean V (mV)": float(np.nanmean(c["V"][:, j0[0]])) if j0 else np.nan,
                 "0 pA sweep, pre-step V (mV)": float(c["V"][:start, j0[0]].mean()) if j0 else np.nan})
rest = pd.DataFrame(rows)
print(rest.round(2).to_string(index=False))
print("\nCells with a 0 pA sweep:", int(rest["0 pA sweep, mean V (mV)"].notna().sum()), "of", len(rest),
      "; nonzero stimulus before the step in", int((rest["max |I| before step (pA)"] > 0).sum()), "cells")
print("Mean V of the 0 pA sweep by group (mV):", rest.groupby("group")["0 pA sweep, mean V (mV)"].mean().round(2).to_dict(),
      "; range", round(rest["0 pA sweep, mean V (mV)"].min(), 2), "to", round(rest["0 pA sweep, mean V (mV)"].max(), 2))

The stimulus series is 0 before the step in every cell, and the 0 pA sweep sits at the same
potential as the pre-step baseline (-61.2 to -76.9 mV), so the DC holding current is not part of
the recorded stimulus and is present throughout every sweep, the 0 pA sweep included. No segment
of the files measures the potential "in the absence of a current injection".

In [ ]:
# Duplicates: identical sweeps across files, and the two CG538 files that share recording number 003.
import hashlib
h = {}
for c in cells:
    for k in range(c["V"].shape[1]):
        h.setdefault(hashlib.md5(np.nan_to_num(c["V"][:, k]).tobytes()).hexdigest(), []).append((c["meta"]["identifier"], k))
dups = [v for v in h.values() if len(v) > 1]
print("Sweeps whose voltage trace is identical to another sweep:", dups if dups else "none")
feat = pd.DataFrame([{**vc.features(c), **vc.quality(c), **{k: c["meta"][k] for k in ["group", "subject", "identifier", "sex", "age"]}} for c in cells])
pair = feat[feat.identifier.str.startswith("20230627_003")]
print("\nThe two CG538 files with recording number 003:")
print(pair[["identifier", "vm", "rin", "tau", "sag", "rheobase", "max_rate", "threshold"]].round(2).to_string(index=False))

No sweep appears twice. The two CG538 files that share recording number 003 (`_NSD` and `_FL`)
differ in sweep length and in every feature (input resistance 192 and 802 MOhm), so they are two
different cells, counted once each.

In [ ]:
# Does the voltage share the clock of the injected current? Contrast windows at the true step onset
# with the same windows shifted 100 ms earlier, for the -60 pA step (hyperpolarization) and for
# spiking during positive steps.
rows = []
for c in cells:
    start, stop, steps, order = vc.step_layout(c)
    dt = 1000 / c["rate"]
    j = [k for k in order if steps[k] == -60][0]
    v = c["V"][:, j]
    def resp(s0):
        return v[s0 + round(20 / dt):s0 + round(80 / dt)].mean() - v[s0 - round(20 / dt):s0].mean()
    sh = round(100 / dt)
    pos = [k for k in order if steps[k] > 0]
    n_in = sum(len(vc.detect_peaks(c["V"][start:stop + 1, k], dt)) for k in pos)
    n_pre = sum(len(vc.detect_peaks(c["V"][start - round(190 / dt):start, k], dt)) for k in pos)
    rows.append({"id": c["meta"]["identifier"], "dV_at_onset": resp(start), "dV_shifted": resp(start - sh),
                 "spikes_in_step_per_s": n_in / (0.5 * len(pos)), "spikes_before_step_per_s": n_pre / (0.19 * len(pos))})
clk = pd.DataFrame(rows)
print(clk.describe().round(2).to_string())
print("\nCells with a hyperpolarization of more than 3 mV at the true onset:", int((clk.dV_at_onset < -3).sum()), "of", len(clk))
print("Cells with more than 1 mV change in the shifted window:", int((clk.dV_shifted.abs() > 1).sum()))
print("Spikes per second during positive steps, all cells:", round(clk.spikes_in_step_per_s.mean(), 2),
      "; in the 190 ms before them:", round(clk.spikes_before_step_per_s.mean(), 2))

In every cell the -60 pA step hyperpolarizes the membrane by 4.8 to 38.5 mV within 20 to 80 ms of
the onset given by the stimulus series, while the same window 100 ms earlier changes by at most
2.0 mV, and spikes occur during positive steps (10.45 per second on average) and never in the
190 ms before them. The voltage and the stimulus share one clock in every file.

## Features and cell selection

The eighteen features of the STAR Methods are computed for every cell (`vpchat.features`). The
Methods' exclusion rules are checked as far as the files allow (`vpchat.quality`): input
resistance under 50 MOhm, resting potential above -50 mV (here the pre-step potential), and a
change of more than 20% in these from the first sweep to the last (input resistance from the
-20 pA test pulse in every sweep). Series resistance is not in the files and cannot be checked.

In [ ]:
print(feat[["group", "subject", "identifier", "vm", "sag", "rin", "tau", "rheobase", "threshold", "amplitude", "width",
            "latency", "upstroke", "downstroke", "max_rate", "ahp_a", "ahp_b", "ahp_code", "ahp_latency", "ahp_width",
            "fi_slope", "adaptation", "cv_isi"]].round(2).to_string(index=False))
print("\nMissing values per feature:", feat.isna().sum()[lambda s: s > 0].to_dict())
q = feat[["group", "identifier", "rin", "vm_max", "rin_tp_first", "rin_tp_last", "rin_tp_change", "vm_first", "vm_last", "vm_change"]]
print("\nExclusion rules:")
print("input resistance (-20 pA step) under 50 MOhm:", int((feat.rin < 50).sum()), "cells; smallest", round(feat.rin.min(), 1))
print("pre-step potential above -50 mV in any sweep:", int((feat.vm_max > -50).sum()), "cells; highest", round(feat.vm_max.max(), 1))
print("test-pulse input resistance changed by more than 20%, first to last sweep:")
print(q[q.rin_tp_change > 0.2].round(2).to_string(index=False))
print("by group:", feat[feat.rin_tp_change > 0.2].group.value_counts().to_dict())
print("pre-step potential changed by more than 20%:", int((feat.vm_change > 0.2).sum()), "cells")

In [ ]:
# Per-sweep test-pulse input resistance (MOhm) and pre-step potential (mV), in file order, for
# every cell; and the 20% rule read as the largest change from the first sweep over all sweeps.
pd.set_option("display.max_colwidth", 400)
print(feat[["group", "identifier", "rin_tp_sweeps", "vm_sweeps"]].to_string(index=False))
print("\nCells whose test-pulse resistance departs by more than 20% from the first sweep in any sweep:",
      feat[feat.rin_tp_maxdev > 0.2].groupby("group").size().to_dict(), "of", feat.groupby("group").size().to_dict())
cv = feat.rin_tp_sweeps.apply(lambda r: np.std(r, ddof=1) / np.mean(r))
print("Sweep-to-sweep CV of the test-pulse resistance within a cell: median", round(cv.median(), 3), "max", round(cv.max(), 3))

No cell breaks the input resistance or resting potential rules. Under the 20% change rule as read
here (the test pulse, first sweep against last), 2 APP, 2 AV and 10 ALL cells would go. That
the APP and AV counts in the dandiset already equal the paper's (12 and 8) suggests the deposited
cells are the ones the paper kept, and that the paper did not apply the change rule this way. The
analysis is therefore run on three cell sets, and each claim reports all three:

- `all`: every cell in the dandiset (24 ALL, 12 APP, 8 AV), the main set;
- `no_FL`: without the three `_FL` files, which leaves 21 ALL cells, the paper's count (from 10
  mice, not 8); that these are the cells the paper left out is a guess;
- `change20`: without the cells whose test-pulse input resistance changed by more than 20%.

The single-sweep test pulse is noisy (the per-sweep table above): read as the largest departure
from the first sweep, the 20% rule would remove 30 of 44 cells, which cannot be the paper's rule.
The first-against-last reading is kept for `change20`, and stays a variant.

Each set is tested by the two readings of the claims file: (a) a cell-level one-way ANOVA and
uncorrected two-tailed Student t-tests for each pair, and (b) the nested design the Methods name
(cells nested in mice nested in group), fitted as a linear mixed model with a random intercept per
mouse, with Wald tests for the group effect and each pair, and the classical nested ANOVA F (group
mean square over mouse-within-group mean square) beside it. In the tables, a pair "X-Y" is Y
minus X, and t is positive when Y is larger. The tables also give `(b) p_t`, the same Wald contrasts
referred to t on the number of mice minus 3 degrees of freedom (`(b) df_t`), since the normal
reference is anticonservative with 4 to 11 mice per group. The STAR Methods (Quantification and
statistical analysis) say "Shapiro-Wilk and Smirnov-Kolmogorov tests were used to assess
normality of the data. Non-parametric tests were conducted if the data failed these tests." The
claims file's readings do not include this rule (a question for a reader, `questions/001544.yaml`),
so the outcomes follow readings (a) and (b), and the tables give beside them the smallest
Shapiro-Wilk p over groups, the Kruskal-Wallis p and two-sided Mann-Whitney U p for each pair
(`MW p`). Each conclusion says whether the non-parametric tests agree.

In [ ]:
sets = {
    "all": feat,
    "no_FL": feat[~feat.identifier.str.endswith("_FL")],
    "change20": feat[~(feat.rin_tp_change > 0.2)],
}
def show(col, sets=sets, **kw):
    t = vc.readings_table(sets, col, **kw)
    print(t.T.to_string(float_format=lambda x: f"{x:.4g}"))
    return t
for k, d in sets.items():
    print(k, d.groupby("group").size().to_dict(), "mice", d.groupby("group").subject.nunique().to_dict())

## fig6f-max-firing-rate

> APP- and AV-odor-activated VP cholinergic neurons both have a higher maximum firing rate than
> home-cage VP cholinergic neurons.

Paper: F(2, 38) = 10.71; ALL vs. APP t(31) = 5.36, p < 0.01; ALL vs. AV t(27) = 2.80, p < 0.05.
Max firing rate is the largest spike count over the 500 ms of any step from 0 to 200 pA, over 0.5 s
(Methods item 16). Reproduced when both pairwise comparisons give p < 0.05 with APP and AV above ALL,
under both test readings.

In [ ]:
t = show("max_rate")
print("\nPer mouse:")
print(vc.per_mouse(feat, "max_rate").to_string(index=False))
print("\nRange over cells (Hz):", feat.max_rate.min(), "to", feat.max_rate.max())

In [ ]:
# Variant: every step in the file, as the code takes them (some ALL cells have steps to 500 pA).
feat_all_steps = pd.DataFrame([{**vc.features(c, use_all_steps=True), **{k: c["meta"][k] for k in ["group", "subject", "identifier"]}} for c in cells])
print("Variant, all steps in the file:")
vc_t = show("max_rate", sets={"all, all steps": feat_all_steps})
fig, ax = plt.subplots(figsize=(4, 3))
for i, g in enumerate(["ALL", "APP", "AV"]):
    y = feat.loc[feat.group == g, "max_rate"]
    ax.plot(np.full(len(y), i) + np.random.default_rng(6).uniform(-0.15, 0.15, len(y)), y, "o", ms=4)
    ax.hlines(y.mean(), i - 0.3, i + 0.3, color="k")
ax.set_xticks([0, 1, 2], ["ALL", "APP", "AV"]); ax.set_ylabel("max firing rate (Hz)")
plt.tight_layout(); plt.show()
print("All points are inside the axes.")

**Result: reproduced.** On all 44 cells, the maximum firing rate is 12.5 Hz in ALL, 50.5 in APP and 49.25 in AV. Reading (a): F(2, 41) = 10.83 (paper 10.71), ALL against APP t(34) = 5.07, p = 1.4e-05 (paper 5.36), ALL against AV t(30) = 3.91, p = 0.00049 (paper 2.80). Reading (b), the mixed model with mice: ALL against APP p = 0.00085, ALL against AV p = 0.0098. Both pairs stay below 0.05 without the `_FL` cells (21 ALL cells; F(2, 38) = 9.80), without the cells whose test-pulse resistance changed by more than 20% (p 0.0011 to 0.024), and with every step in the file as the code takes them (p 6.6e-05 to 0.025). Per mouse, the means range from 4 to 87 Hz in APP and 13 to 90 Hz in AV, against 4 to 29 Hz in ALL; 4 of 6 APP mice (CC328, CC330, CC356, CC396) and 2 of 4 AV mice (CC318, CC399) are above every ALL mouse, the highest being CG355 at 29.00 Hz. With the mixed-model contrasts referred to t on 18 df, p = 0.0037 and 0.019. The non-parametric tests the Methods prescribe when normality fails agree (Shapiro-Wilk p = 0.00023; Kruskal-Wallis p = 0.0030; Mann-Whitney ALL against APP p = 0.0023, ALL against AV p = 0.022).

## figs12a-sag-higher

> APP- and AV-odor-activated VP cholinergic neurons both have a larger sag potential than home-cage
> VP cholinergic neurons.

Sag from the -60 pA step, as in the code (20 ms median filter, maximum of the second half of the step
minus the minimum of the first half). Reproduced when APP and AV each exceed ALL at p < 0.05, under
the two test readings.

In [ ]:
t = show("sag")
print("\nSag step used:", feat.sag_step.unique().tolist(), "pA")
print("\nPer mouse:")
print(vc.per_mouse(feat, "sag").to_string(index=False))
print("\nRange over cells (mV):", round(feat.sag.min(), 2), "to", round(feat.sag.max(), 2), "; cells with sag 0:", int((feat.sag == 0).sum()))

In [ ]:
# The -60 pA response reaches the recording floor (-105.6 mV) for 10 ms or more in five cells, which
# truncates its minimum. Variant: the code's intended rule, the next step up that is not saturated.
print("Cells whose -60 pA response is saturated:")
print(feat.loc[feat.sag_saturated, ["group", "identifier", "sag"]].round(2).to_string(index=False))
feat_sat = pd.DataFrame([{**vc.features(c, skip_saturated_sag=True), **{k: c["meta"][k] for k in ["group", "subject", "identifier"]}} for c in cells])
print("\nThe same cells with the next unsaturated step:")
print(feat_sat.loc[feat_sat.sag_saturated, ["group", "identifier", "sag", "sag_step"]].round(2).to_string(index=False))
print()
_ = show("sag", sets={"all, unsaturated step": feat_sat, "no_FL, unsaturated step": feat_sat[~feat_sat.identifier.str.endswith("_FL")]})

# The code's saturation test (sagpotential.m) on the -60 pA sweep of every cell.
rows = []
for c in cells:
    start, stop, steps, order = vc.step_layout(c)
    j = [k for k in order if steps[k] == -60][0]
    v = c["V"][start:stop + 1, j]
    rows.append({"id": c["meta"]["identifier"], "group": c["meta"]["group"], **vc.code_saturation_test(v),
                 "min (mV)": float(np.nanmin(v)), "samples at -105.62 mV": int((np.abs(v + 105.62) < 0.01).sum()),
                 "saturated (10 ms at min)": vc.saturated(v)})
sat = pd.DataFrame(rows)
print("\nThe code's test: largest mode count", sat.mode_count.max(), "; fires in", int(sat.code_fires.sum()), "cells")
print("Counting unchanged samples against 1000 (what the test seems meant to do) fires in",
      int(sat.intended_fires.sum()), "cells:")
print(sat[sat.intended_fires | sat["saturated (10 ms at min)"]].round(2).to_string(index=False))
print("\nIs -105.62 mV a floor over all files? Global minimum over every sweep of every cell:",
      round(min(float(np.nanmin(c["V"])) for c in cells), 2), "mV; cells with samples at -105.62 mV:",
      {c["meta"]["identifier"]: int((np.abs(c["V"] + 105.62) < 0.01).sum()) for c in cells if (np.abs(c["V"] + 105.62) < 0.01).any()})

# Samples below the -105.62 mV floor, per file, and how many sit at the global minimum.
gmin = min(float(np.nanmin(c["V"])) for c in cells)
below = {c["meta"]["identifier"]: int((c["V"] < -105.7).sum()) for c in cells if (c["V"] < -105.7).any()}
print("Samples below -105.7 mV, per cell:", below)
print("Samples within 0.01 mV of the global minimum", round(gmin, 2), "mV, per cell:",
      {c["meta"]["identifier"]: int((np.abs(c["V"] - gmin) < 0.01).sum()) for c in cells if (np.abs(c["V"] - gmin) < 0.01).any()})


In [ ]:
# The -60 pA sweeps of the five saturated cells.
fig, axs = plt.subplots(1, 5, figsize=(14, 2.6), sharey=True)
for ax, (_, r) in zip(axs, feat[feat.sag_saturated].iterrows()):
    c = next(c for c in cells if c["meta"]["identifier"] == r.identifier)
    start, stop, steps, order = vc.step_layout(c)
    j = [k for k in order if steps[k] == -60][0]
    t = np.arange(c["V"].shape[0]) / c["rate"] * 1000 - start / c["rate"] * 1000
    ax.plot(t, c["V"][:, j], lw=0.6)
    ax.axhline(-105.62, color="k", ls=":", lw=0.8)
    ax.set_xlim(-100, 700)
    ax.set_title(f"{r.group} {r.identifier}", fontsize=8)
    ax.set_xlabel("ms from step onset")
axs[0].set_ylabel("mV")
plt.tight_layout(); plt.show()
print("Axes: -100 to 700 ms; y spans each trace's range (shared). Sweep minima (mV):",
      {r.identifier: round(float(np.nanmin(next(c for c in cells if c["meta"]["identifier"] == r.identifier)["V"])), 2)
       for _, r in feat[feat.sag_saturated].iterrows()})

In [ ]:
# Variant: leave out the five cells whose -60 pA response is saturated, since their sag is measured
# on a clipped trace and a smaller step gives a smaller sag by construction.
_ = show("sag", sets={k + ", without saturated": d[~d.sag_saturated] for k, d in sets.items()})

**Result: not reproduced.** On all 44 cells, sag is 3.22 mV in ALL, 6.68 in APP and 4.90 in AV. AV does not exceed ALL under either reading (t = 1.77, p = 0.087; mixed model p = 0.24, 0.25 with t on 18 df), so the criterion, which needs both groups, fails. APP exceeds ALL by both parametric readings (p = 0.011; mixed model p = 0.0041, 0.010 with t on 18 df), but not by the non-parametric tests the Methods prescribe when normality fails (Shapiro-Wilk p = 0.0016; Kruskal-Wallis p = 0.12; Mann-Whitney ALL against APP p = 0.10, ALL against AV p = 0.094). The outcome depends on the cell set. Without the three `_FL` cells, reading (a) gives both pairs (p = 0.0036 and 0.0072) while reading (b) misses AV (p = 0.12); without the cells whose test-pulse resistance changed by more than 20%, neither pair is significant (p = 0.18 and 0.086). Five cells (four APP, one ALL) sit at -105.62 mV for 45 to 463 ms of the -60 pA step, which clips their minimum, so their sag is underestimated; they are flat at that value. It is not a floor in every file: one other APP cell (20221214_001) has 7 samples at -105.62 mV and 1735 below -105.7 mV, down to -112.88 mV (a single sample at the minimum). The code's saturation test never fires (its mode count is 1 in every cell); counting unchanged samples against 1000 would fire in six cells, three of them not saturated, and miss two of the five saturated ones, so the test as it seems meant would not separate them either. With the next unsaturated step for the five cells, neither pair is significant on all cells (p = 0.053 and 0.085). Leaving the five cells out, APP exceeds ALL on all cells (p = 0.016; mixed model p = 0.017) and AV still does not (p = 0.092; mixed model p = 0.13). The outcome follows readings (a) and (b): in the main set neither gives AV above ALL, and the criterion needs both groups, so the outcome is not reproduced. The rank tests and the unsaturated-step variant do not decide it; they show that the APP half is not robust either. The outcome rests on the dandiset's 24 ALL cells, since without the `_FL` cells reading (a) would give both pairs.

## fig6g-av-more-hyperpolarized

> AV-odor-activated VP cholinergic neurons have a more hyperpolarized membrane potential than
> APP-odor-activated ones.

Paper: F(2, 38) = 11.39, p < 0.001; APP vs. AV t(18) = 2.72, p < 0.05. Membrane potential reading
(a), the code's baselineVm (mean over sweeps of the voltage 10 to 1 ms before the step). Reading (b),
a segment with no injected current, does not exist in the files (Data check). Reproduced when the
ANOVA and the APP against AV comparison give p < 0.05 with AV more negative under every reading run.

In [ ]:
t = show("vm")
print("\nPer mouse:")
print(vc.per_mouse(feat, "vm").to_string(index=False))
print("\nRange over cells (mV):", round(feat.vm.min(), 2), "to", round(feat.vm.max(), 2))
print("Spread of the pre-step potential across sweeps within a cell (SD, mV): median",
      round(feat.vm_sweep_sd.median(), 2), "max", round(feat.vm_sweep_sd.max(), 2))

**Result: not reproduced.** The pre-step membrane potential (the code's baselineVm) is -66.51 mV in ALL, -66.59 in APP and -67.25 in AV: AV is 0.66 mV more negative than APP, but the APP against AV comparison gives t(18) = -0.53, p = 0.60 (paper t(18) = 2.72, p < 0.05), and the ANOVA F(2, 41) = 0.29, p = 0.75 (paper F = 11.39, p < 0.001). The mixed model gives p = 0.76 for the pair and 0.95 for the group, and the APP against AV p is 0.34 or more in every cell set and reading. Cell values range from -73.5 to -62.0 mV. Reading (b), the potential without injected current, cannot be run: the files hold no segment without the DC holding current, which keeps every cell near -65 mV, so this measure cannot show differences in resting potential. Reading (a) is what the claims file defines from the code, and the outcome follows its criterion; but a potential measured under a DC injection meant to hold every cell at -65 mV can hardly differ by group, so it may not be what Figure 6G plots, and the failure here says little about the paper's resting potential. The 0 pA sweeps, also under the holding current, give no other measure (Data check). The non-parametric tests agree (Kruskal-Wallis p = 0.89; Mann-Whitney APP against AV p = 0.79).

## fig6h-av-smaller-ahp

> AV-odor-activated VP cholinergic neurons have a smaller afterhyperpolarization amplitude than
> APP-odor-activated ones.

Paper: F(2, 38) = 4.53, p < 0.05; APP vs. AV t = 2.74, p < 0.05. AHP amplitude after the first
spike at rheobase, threshold at dV/dt 20 mV/ms, readings (a) threshold minus the minimum within
100 ms and (b) threshold minus the voltage 100 ms after threshold; the code's definition is a
variant. Reproduced when the ANOVA and the APP against AV comparison give p < 0.05 with AV smaller
under every Methods reading.

In [ ]:
print("Reading (a), minimum within 100 ms:")
ta = show("ahp_a")
print("\nReading (b), voltage at 100 ms (cells whose first spike is less than 100 ms before the end of the step have none):")
tb = show("ahp_b")
print("cells without a value:", feat[feat.ahp_b.isna()].groupby("group").size().to_dict())
print("Reading (a) window cut short by the end of the step (under 100 ms):")
print(feat.loc[feat.ahp_window_ms < 100, ["group", "identifier", "latency", "ahp_window_ms", "ahp_a", "ahp_latency"]].round(2).to_string(index=False))
print("\nVariant, the code's definition (afterhyperpolarization.m):")
tc = show("ahp_code")
print("\nPer mouse, reading (a):")
print(vc.per_mouse(feat, "ahp_a").to_string(index=False))

In [ ]:
# Threshold at the code's 10 mV/ms in place of the Methods' 20 mV/ms.
feat10 = pd.DataFrame([{**vc.features(c, dvdt_threshold=vc.CODE_DVDT), **{k: c["meta"][k] for k in ["group", "subject", "identifier"]}} for c in cells])
print("Threshold at 10 mV/ms, reading (a):")
_ = show("ahp_a", sets={"all, 10 mV/ms": feat10})
print("\nThreshold at 10 mV/ms, reading (b):")
_ = show("ahp_b", sets={"all, 10 mV/ms": feat10})

**Result: not reproduced.** Reading (a), threshold minus the minimum within 100 ms: APP 16.67 and AV 15.61 mV, APP against AV p = 0.68, ANOVA p = 0.55. Reading (b), threshold minus the voltage at 100 ms: AV is larger than APP (12.58 against 9.45 mV, p = 0.18), the opposite direction, and the ANOVA (p = 0.049) comes from ALL being larger than APP. The mixed model gives p = 0.89 and 0.41 for the APP against AV pair. The code's definition gives AV smaller than APP (9.45 against 15.61 mV) without significance (t = -1.85, p = 0.081; mixed model p = 0.27; ANOVA p = 0.27). Threshold at 10 mV/ms changes none of this (reading (a) pair p = 0.64). In the same two cells (one APP, one ALL) the window of reading (a) is cut by the end of the step to 76.8 and 53.3 ms; their minima fall at 8.7 and 37.0 ms, inside the cut window, so the truncation does not change their values. The non-parametric tests agree (APP against AV Mann-Whitney p = 0.91 for reading (a), 0.24 for reading (b), 0.083 for the code's definition).

## fig6-app-less-excitable

> APP-odor-activated VP cholinergic neurons are less excitable than AV-odor-activated ones, with
> different action potential profiles.

Readings of excitability from the claims file: (a) spike latency at rheobase (Methods item 9, step
onset to threshold) longer in APP, and (b) rheobase (item 5) higher in APP, each by the two test
readings at p < 0.05. Reproduced when every reading gives a significant difference in that direction,
not reproduced when none does.

In [ ]:
print("Spike latency at rheobase (ms), step onset to threshold:")
tl = show("latency")
print("\nRheobase (pA):")
tr = show("rheobase")
print("\nVariant, latency to the peak (code):")
_ = show("latency_peak")
print("\nVariant, latency with threshold at 10 mV/ms:")
_ = show("latency", sets={"all, 10 mV/ms": feat10})
print("\nPer mouse, latency:")
print(vc.per_mouse(feat, "latency").to_string(index=False))
print("\nPer mouse, rheobase:")
print(vc.per_mouse(feat, "rheobase").to_string(index=False))

**Result: not reproduced.** Latency to the first spike at rheobase is longer in APP than in AV (153.2 against 74.5 ms), but not significantly (t(18) = -1.90, p = 0.074; mixed model p = 0.097); measured to the peak or with threshold at 10 mV/ms it is the same (p = 0.074). Rheobase goes the other way: APP 35.0 against AV 51.25 pA (p = 0.26; mixed model p = 0.52). No reading gives a significant difference in the stated direction, in any cell set (latency p 0.074 or more; rheobase is higher in APP only without the changed cells, 36 against 35 pA, p = 0.92). The non-parametric tests agree for the APP against AV pair (Mann-Whitney p = 0.057 for latency, 0.37 for rheobase).

## figs12-other-features-shared

> Most other passive and active electrophysiological features do not differ between home-cage and
> odor-activated VP cholinergic neurons; the Figure S12 legend names ten that are similar across
> ALL, APP and AV (threshold, upstroke, spike amplitude, spike latency, AHP latency, AHP width, CV
> of ISIs, rheobase, tau, adaptation index).

The claim was amended in answer to a question from this attempt (`questions/001544.yaml`): it was
first judged on fourteen features (the eighteen of the Methods less max firing rate, sag, membrane
potential and AHP amplitude), and is now judged on the ten the Figure S12 legend names as similar
(panels F to O). Each is tested by the cell-level one-way ANOVA over the three groups at alpha 0.05
(reading (a)). Reproduced when more than half (six or more) show no difference. The other four of
the fourteen (input resistance, spike width, downstroke, f-I slope) are now the claim
figs12-app-differs-from-all, below; the earlier count over fourteen is shown as a variant.

In [ ]:
s12 = ["threshold", "upstroke", "amplitude", "latency", "ahp_latency", "ahp_width", "cv_isi", "rheobase", "tau",
       "adaptation"]
def feature_table(features):
    rows = []
    for name, d in sets.items():
        for col in features:
            a = vc.cell_tests(d, col)
            b = vc.nested_tests(d, col)
            x = d.dropna(subset=[col])
            ssb = sum(len(g) * (g[col].mean() - x[col].mean()) ** 2 for _, g in x.groupby("group"))
            rows.append({"cells": name, "feature": col, **{f"mean {k}": a["mean"][k] for k in ["ALL", "APP", "AV"]},
                         "n": "/".join(str(a["n"][k]) for k in ["ALL", "APP", "AV"]),
                         "F": a["F"], "p": a["p_anova"], "eta2": ssb / ((x[col] - x[col].mean()) ** 2).sum(),
                         "p ALL-APP": a["ALL-APP"]["p"], "p ALL-AV": a["ALL-AV"]["p"],
                         "mixed p": b["p_group"], "Kruskal p": a["p_kruskal"]})
    return pd.DataFrame(rows)
oth = feature_table(s12)
print(oth.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print("\nFeatures with no difference, of 10 (one-way ANOVA p >= 0.05; nested reading (b), mixed model p >= 0.05):")
for name in sets:
    sub = oth[oth.cells == name]
    print(f"{name:9s} ANOVA {int((sub.p >= 0.05).sum())}, mixed model {int((sub['mixed p'] >= 0.05).sum())};"
          f" differ by ANOVA: {sub.loc[sub.p < 0.05, 'feature'].tolist()}")
print("\nNon-parametric (Kruskal-Wallis p >= 0.05), the Methods' rule when normality fails:")
for name in sets:
    sub = oth[oth.cells == name]
    print(f"{name:9s} {int((sub['Kruskal p'] >= 0.05).sum())} of 10; differ: {sub.loc[sub['Kruskal p'] < 0.05, 'feature'].tolist()}")

In [ ]:
# Variant: the fourteen features of the claim before it was amended (the ten above, with input resistance,
# spike width, downstroke and f-I slope), for comparison with the earlier outcome.
oth14 = pd.concat([oth, feature_table(["rin", "width", "downstroke", "fi_slope"])])
for name in sets:
    sub = oth14[oth14.cells == name]
    print(f"{name:9s} no difference in {int((sub.p >= 0.05).sum())} of 14 by ANOVA, "
          f"{int((sub['mixed p'] >= 0.05).sum())} of 14 by the mixed model")

**Result: reproduced.** On all 44 cells, 9 of the ten features the Figure S12 legend names as similar show no difference by the one-way ANOVA (p >= 0.05): threshold, upstroke, spike amplitude, spike latency, AHP width, CV of ISIs, rheobase, tau and adaptation index. AHP latency differs (p = 0.012; APP 24.9 ms against ALL 48.6 and AV 56.8). The count is 7 of 10 without the `_FL` cells (rheobase p = 0.045 and adaptation index p = 0.040 also differ) and 10 of 10 without the cells whose resistance changed by more than 20%; the mixed model gives 9, 7 and 10. Every cell set and both tests give more than half, so the outcome does not rest on which ALL cells are used. By the Kruskal-Wallis test, the non-parametric reading the Methods prescribe when normality fails, the count is 7 of 10 on all cells (spike latency p = 0.025, AHP latency 0.016 and adaptation index 0.032 differ), 6 of 10 without the `_FL` cells and 9 of 10 without the changed cells, still more than half in each. Adaptation index, marked with an asterisk in the figure, is near the threshold on all cells (p = 0.056). Before the claim was amended, the count over fourteen features was 9 of 14 on all cells and 7 of 14 without the `_FL` cells, so the earlier outcome depended on the cell set.

## figs12-app-differs-from-all

> APP-odor-activated VP cholinergic neurons differ from home-cage VP cholinergic neurons in input
> resistance, action potential width, action potential downstroke and f-I slope.

A claim added to the claims file in answer to this attempt's question. The four features by the
Methods definitions (input resistance from the -20 pA step), with the two test readings of
fig6f-max-firing-rate; ALL against APP is judged at p < 0.05 and its direction reported.
Reproduced when all four differ under both readings, partially reproduced when some do, not
reproduced when none does; blocked as methods_underspecified when the readings disagree.

In [ ]:
units = {"rin": "MOhm", "width": "ms", "downstroke": "mV/ms", "fi_slope": "Hz/pA"}
app = {}
for col, u in units.items():
    print(f"{col} ({u}):")
    app[col] = show(col)
    print()
summary = pd.DataFrame([{"cells": name, "feature": col,
                         "APP - ALL": app[col].at[name, "mean APP"] - app[col].at[name, "mean ALL"],
                         "(a) p": app[col].at[name, "(a) p ALL-APP"], "(b) p": app[col].at[name, "(b) p ALL-APP"],
                         "(b) p_t": app[col].at[name, "(b) p_t ALL-APP"], "MW p": app[col].at[name, "MW p ALL-APP"]}
                        for name in sets for col in units])
summary["differs (a)"] = summary["(a) p"] < 0.05
summary["differs (b)"] = summary["(b) p"] < 0.05
summary["differs (b) t df"] = summary["(b) p_t"] < 0.05
summary["differs MW"] = summary["MW p"] < 0.05
print(summary.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print("\nFeatures where ALL against APP gives p < 0.05, of 4:")
print(summary.groupby("cells")[["differs (a)", "differs (b)", "differs (b) t df", "differs MW"]].sum().to_string())

In [ ]:
print("Per mouse (mean of each mouse's cells):")
for col, u in units.items():
    pm = vc.per_mouse(feat, col)
    print(f"{col} ({u}):", {g: f"{s['mean'].min():.3g} to {s['mean'].max():.3g}" for g, s in pm.groupby("group")})
print("\nRange over cells:", {col: (round(feat[col].min(), 3), round(feat[col].max(), 3)) for col in units})
print("Missing values:", {col: int(feat[col].isna().sum()) for col in units})

**Result: reproduced.** On all 44 cells, APP differs from ALL in each of the four features under both test readings. Input resistance is higher in APP (548 against 335.5 MOhm; t = 2.78, p = 0.0089; mixed model p = 0.022), spike width is shorter (0.915 against 1.52 ms; p = 0.00037; mixed model p = 0.00082), the downstroke is steeper (-74.7 against -41.2 mV/ms; p = 0.0015; mixed model p = 0.026) and the f-I slope is larger (0.329 against 0.108 Hz/pA; p = 2.8e-05; mixed model p = 0.0012). The same holds without the `_FL` cells (p from 9.1e-05 to 0.041). Without the cells whose resistance changed by more than 20%, input resistance no longer differs (p = 0.20 and 0.19) and the downstroke differs only by the cell-level test (p = 0.029; mixed model p = 0.11), so on that set the claim would be partially reproduced; the outcome stated rests on the dandiset's 24 ALL cells. With the mixed-model contrasts referred to t on the number of mice minus 3 df, the downstroke p rises to 0.039 on all cells and 0.057 without the `_FL` cells, so that feature is marginal; the other three stay below 0.05 in both sets. The non-parametric tests agree on all cells (Mann-Whitney ALL against APP p = 0.0051, 0.00051, 0.0046 and 0.00066).

## Claims not tested

The dandiset holds only the patch-clamp recordings. The claims below need data it does not hold:
Y-maze tracking (fig1c-app-approach, fig1c-av-avoidance, figs1-locomotion-unchanged,
fig2c-hm4di-app-reversal, fig2e-hm4di-av-unchanged, fig5c-adcd-app-reversal,
fig5e-adcd-av-unchanged), cell counts from immunohistochemistry (fig1f-cfos-total,
fig1g-chat-cfos, fig3-same-odor-reactivation, fig4d-different-odor-no-reactivation,
figs5-activation-counts, figs7-ram-replication, figs9-valence-specific, figs11-intermingled,
fig7d-bla-projecting-av), fiber photometry (fig1k-app-calcium-auc, fig1l-av-calcium-auc,
fig1mn-calcium-max-z) and the Imaris reconstructions of neurobiotin-filled cells
(fig6j-convex-hull-area, fig6kl-dendrite-area-volume, fig6m-sholl-proximal, fig6n-sholl-reach,
figs13-other-morphometrics). The paper's data statement says "All data reported in this paper will be
shared by the lead contact upon request" and names only its two code repositories; a search of DANDI
for the paper's title and senior author finds no other dandiset of this paper.

In [ ]:
print(f"notebook ran in {time.time() - started:.0f} s")

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| fig6f-max-firing-rate | reproduced |
| figs12a-sag-higher | not reproduced (assumed) |
| figs12-other-features-shared | reproduced |
| figs12-app-differs-from-all | reproduced (assumed) |
| fig6g-av-more-hyperpolarized | not reproduced |
| fig6h-av-smaller-ahp | not reproduced |
| fig6-app-less-excitable | not reproduced |
| 24 other claims (behaviour, histology, photometry, morphology) | blocked: data missing |

How the analysis differs from the paper's: The features follow the STAR Methods, with the
paper's code (v1.0.0) where the Methods are silent; the code's own definitions of threshold (10
mV/ms), latency (to the peak) and AHP amplitude are shown as variants. The paper analysed 21 ALL
cells from 8 mice; the dandiset holds 24 from 11 and does not say which were left out, so all 24
are used, with the 21 cells left when the `_FL` files are set aside as a variant. The exclusion
rule on a 20% change could only be read through the test pulse, and is a variant. The nested ANOVA
of the Methods is fitted as a mixed model with a random intercept per mouse, with Wald tests
against the normal and against t on the number of mice minus 3 df. The Methods' switch to
non-parametric tests when normality fails is not in the claims file's readings; Kruskal-Wallis
and Mann-Whitney results are shown beside them; every outcome follows the claims file's readings, and the
rank tests would change none of them. Membrane potential
can only be read under the DC holding current.

What was found about the dataset: The stimulus is in picoamperes though stated in amperes.
`bias_current` and `bridge_balance` hold the same value in every file, and the holding current is
not recorded. The ALL group has 24 cells from 11 mice where the paper has 21 from 8, and five of its
mice are younger than the 8 to 12 weeks of the Methods. The APP mice are 4 male and 2 female where
the legend says 3 and 3. Five recordings saturate at -105.6 mV during the -60 pA step, and the code's saturation test
cannot detect it. Every sweep
starts at time 0, so the interval between steps is not recorded. Only the patch-clamp data are in
the dandiset; the paper deposits none of its other data.